#  Retrieval Evaluation on Validation Claims

Goal: take the six chunked corpora produced on Day 2 and measure how well each one retrieves the *gold evidence document* for the 338 SciFact validation claims.

For every chunking strategy we:

1. Embed all chunks with `BAAI/bge-small-en-v1.5`.
2. Build an in-memory FAISS index (cosine similarity via inner product on L2-normalized vectors).
3. Embed each validation claim, retrieve the top-K chunks, and **roll chunks up to their `doc_id`** before scoring (multiple chunks of the same doc collapse to one rank).
4. Compute Hit Rate@K and MRR@K for K = 1, 3, 5, 10.

The output is a single comparison table that tells us which chunking strategy actually helps retrieval — the answer Day 2 was building toward.

**Inputs:**
- `claims_val.json` (from Day 1)
- `chunks/chunks_*.json` (six files from Day 2)

**Outputs:**
- `eval/retrieval_metrics.csv`
- `eval/retrieval_metrics.json`


## 0. Install dependencies

Run once. `faiss-cpu` is fine here — the corpus is small (~5K docs) so GPU FAISS isn't needed; the embedding step is the expensive part and uses GPU automatically if available.

In [2]:
%pip install -q sentence-transformers faiss-cpu pandas tqdm numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 118.5 MB/s eta 0:00:00


In [3]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

import faiss
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda


## 1. Paths and config

`K_VALUES` controls which Hit Rate@K and MRR@K we report. `MAX_K` is what we actually retrieve from FAISS — we always retrieve `max(K_VALUES)` and slice down for smaller K, so one search per claim covers every metric.

In [4]:
CHUNK_DIR = Path(".")           # files are flat in /content
EVAL_DIR = Path("eval")
EVAL_DIR.mkdir(exist_ok=True)

CLAIMS_PATH = Path("claims_val.json")

CHUNK_FILES = {
    "Document-level":  CHUNK_DIR / "chunks_document.json",
    "Sentence-level":  CHUNK_DIR / "chunks_sentence.json",
    "Fixed 128":       CHUNK_DIR / "chunks_fixed_128.json",
    "Recursive 128":   CHUNK_DIR / "chunks_recursive_128.json",
    "Recursive 256":   CHUNK_DIR / "chunks_recursive_256.json",
    "Semantic":        CHUNK_DIR / "chunks_semantic.json",
}

EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"
K_VALUES = [1, 3, 5, 10]
MAX_K = max(K_VALUES)

QUERY_INSTRUCTION = "Represent this sentence for searching relevant passages: "
BATCH_SIZE = 64

## 2. Load validation claims

Day 1 already filtered out claims with no `evidence_doc_id`, so every claim here has a ground-truth document we can score against.

In [6]:
with open(CLAIMS_PATH, "r") as f:
    claims = json.load(f)

print(f"Loaded {len(claims)} validation claims")
print(f"Sample claim: {claims[0]['claim']!r}")
print(f"Sample evidence_doc_id: {claims[0]['evidence_doc_id']}")

# Pre-extract query texts and gold doc IDs (as strings, to match chunk doc_ids)
query_texts = [c["claim"] for c in claims]
gold_doc_ids = [str(c["evidence_doc_id"]) for c in claims]

Loaded 338 validation claims
Sample claim: '1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants.'
Sample evidence_doc_id: 14717500


## 3. Load embedding model once

We load `bge-small-en-v1.5` once and reuse it across all six strategies. Loading it inside the per-strategy loop would re-download / re-initialize it six times.

In [7]:
model = SentenceTransformer(EMBED_MODEL_NAME, device=device)
print(f"Model loaded: {EMBED_MODEL_NAME}")
print(f"Embedding dim: {model.get_sentence_embedding_dimension()}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded: BAAI/bge-small-en-v1.5
Embedding dim: 384


/tmp/ipykernel_2160/1066904624.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dim: {model.get_sentence_embedding_dimension()}")


## 4. Embed the validation claims once

Claim embeddings are the same for every strategy — only the chunk set changes. So we encode the 338 queries once, normalize for cosine, and reuse.

In [8]:
prefixed_queries = [QUERY_INSTRUCTION + q for q in query_texts]

query_embeddings = model.encode(
    prefixed_queries,
    batch_size=BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype("float32")

print(f"Query embeddings shape: {query_embeddings.shape}")

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Query embeddings shape: (338, 384)


## 5. Core helpers: build, search, score

- `build_index(chunks)` — embeds chunk texts and builds a FAISS `IndexFlatIP` (cosine via normalized vectors).
- `evaluate(index, doc_ids_in_order)` — searches each query, **deduplicates retrieved chunks by `doc_id` keeping the best rank**, then computes Hit Rate@K and MRR@K.

The dedup step is the important detail: with sentence-level chunking a single document can contribute many chunks. If the gold doc shows up at chunk-rank 1 *and* chunk-rank 4, that should count as doc-rank 1 — not double-counting and not letting a worse rank win.

In [9]:
def build_index(chunks):
    """Embed chunk texts and return (faiss_index, doc_ids_in_chunk_order)."""
    texts = [c["text"] for c in chunks]
    doc_ids = [str(c["doc_id"]) for c in chunks]

    embeddings = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype("float32")

    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)

    return index, doc_ids


def evaluate(index, chunk_doc_ids, query_embs, gold_ids, k_values, max_k):
    """Run search, dedup chunks to docs by best rank, return metrics dict."""
    # Over-retrieve so that after doc-level dedup we still have enough unique docs
    # to populate the largest K. A factor of 5x is generous for sentence-level
    # (avg ~5 sentences/doc) and harmless for coarser strategies.
    search_k = max_k * 5
    search_k = min(search_k, index.ntotal)

    _, indices = index.search(query_embs, search_k)

    metrics = {f"hit@{k}": 0 for k in k_values}
    metrics.update({f"mrr@{k}": 0.0 for k in k_values})

    n_queries = len(gold_ids)

    for q_idx, gold in enumerate(gold_ids):
        # Walk retrieved chunks in order; record doc rank the first time we see each doc
        seen_docs = []
        for chunk_idx in indices[q_idx]:
            doc_id = chunk_doc_ids[chunk_idx]
            if doc_id not in seen_docs:
                seen_docs.append(doc_id)
            if len(seen_docs) >= max_k:
                break

        # Find rank of the gold doc within the deduplicated doc list (1-indexed)
        if gold in seen_docs:
            doc_rank = seen_docs.index(gold) + 1
        else:
            doc_rank = None

        for k in k_values:
            if doc_rank is not None and doc_rank <= k:
                metrics[f"hit@{k}"] += 1
                metrics[f"mrr@{k}"] += 1.0 / doc_rank

    # Convert counts to rates
    for k in k_values:
        metrics[f"hit@{k}"] = metrics[f"hit@{k}"] / n_queries
        metrics[f"mrr@{k}"] = metrics[f"mrr@{k}"] / n_queries

    return metrics

## 6. Run evaluation on every chunking strategy

This is the heavy cell. For each of the six strategies we embed all its chunks, build a FAISS index, run all 338 queries through it, and record metrics. Total runtime ≈ 5–15 min on GPU, longer on CPU (sentence-level is the slowest because it has the most chunks).

In [10]:
results = []

for strategy_name, chunk_path in CHUNK_FILES.items():
    print(f"\n{'='*60}")
    print(f"Strategy: {strategy_name}")
    print(f"{'='*60}")

    with open(chunk_path, "r") as f:
        chunks = json.load(f)

    print(f"Chunks: {len(chunks)}")

    print("Embedding chunks and building FAISS index...")
    index, chunk_doc_ids = build_index(chunks)

    print("Evaluating on validation claims...")
    metrics = evaluate(
        index=index,
        chunk_doc_ids=chunk_doc_ids,
        query_embs=query_embeddings,
        gold_ids=gold_doc_ids,
        k_values=K_VALUES,
        max_k=MAX_K,
    )

    row = {
        "Strategy": strategy_name,
        "Num chunks": len(chunks),
    }
    row.update({f"Hit@{k}": metrics[f"hit@{k}"] for k in K_VALUES})
    row.update({f"MRR@{k}": metrics[f"mrr@{k}"] for k in K_VALUES})

    results.append(row)

    # Print a per-strategy summary
    summary_bits = [f"Hit@{k}={metrics[f'hit@{k}']:.3f}" for k in K_VALUES]
    summary_bits += [f"MRR@{k}={metrics[f'mrr@{k}']:.3f}" for k in K_VALUES]
    print("  " + "  ".join(summary_bits))

    # Free the FAISS index before the next strategy so memory doesn't pile up
    del index



Strategy: Document-level
Chunks: 5183
Embedding chunks and building FAISS index...


Batches:   0%|          | 0/81 [00:00<?, ?it/s]

Evaluating on validation claims...
  Hit@1=0.740  Hit@3=0.858  Hit@5=0.893  Hit@10=0.956  MRR@1=0.740  MRR@3=0.792  MRR@5=0.800  MRR@10=0.809

Strategy: Sentence-level
Chunks: 45952
Embedding chunks and building FAISS index...


Batches:   0%|          | 0/718 [00:00<?, ?it/s]

Evaluating on validation claims...
  Hit@1=0.737  Hit@3=0.858  Hit@5=0.929  Hit@10=0.967  MRR@1=0.737  MRR@3=0.792  MRR@5=0.807  MRR@10=0.813

Strategy: Fixed 128
Chunks: 17400
Embedding chunks and building FAISS index...


...

Evaluating on validation claims...
  Hit@1=0.749  Hit@3=0.870  Hit@5=0.923  Hit@10=0.950  MRR@1=0.749  MRR@3=0.802  MRR@5=0.813  MRR@10=0.817

Strategy: Recursive 256
Chunks: 9905
Embedding chunks and building FAISS index...


Batches:   0%|          | 0/155 [00:00<?, ?it/s]

Evaluating on validation claims...
  Hit@1=0.737  Hit@3=0.870  Hit@5=0.914  Hit@10=0.970  MRR@1=0.737  MRR@3=0.795  MRR@5=0.806  MRR@10=0.813

Strategy: Semantic
Chunks: 10594
Embedding chunks and building FAISS index...


Batches:   0%|          | 0/166 [00:00<?, ?it/s]

Evaluating on validation claims...
  Hit@1=0.734  Hit@3=0.849  Hit@5=0.899  Hit@10=0.953  MRR@1=0.734  MRR@3=0.787  MRR@5=0.799  MRR@10=0.806


In [11]:
import os
print("Working directory:", os.getcwd())
!ls

Working directory: /content
chunks_document.json	   chunks_semantic.json  eval
chunks_fixed_128.json	   chunks_sentence.json  sample_data
chunks_recursive_128.json  claims_val.json
chunks_recursive_256.json  corpus.json


## 7. Comparison table

The headline result. Higher is better on every column. Hit@1 = "did the very top doc match?", MRR@10 = "averaged inverse rank if it appeared in the top 10". MRR rewards strategies that put the gold doc at rank 1 vs. rank 10.

In [12]:
results_df = pd.DataFrame(results)

# Round metric columns for readability; leave counts as ints
metric_cols = [c for c in results_df.columns if c.startswith(("Hit@", "MRR@"))]
display_df = results_df.copy()
display_df[metric_cols] = display_df[metric_cols].round(3)

display_df

,Strategy,Num chunks,Hit@1,Hit@3,Hit@5,Hit@10,MRR@1,MRR@3,MRR@5,MRR@10
0,Document-level,5183,0.740,0.858,0.893,0.956,0.740,0.792,0.800,0.809
1,Sentence-level,45952,0.737,0.858,0.929,0.967,0.737,0.792,0.807,0.813
2,Fixed 128,17400,0.740,0.834,0.891,0.950,0.740,0.783,0.796,0.804
3,Recursive 128,18638,0.749,0.870,0.923,0.950,0.749,0.802,0.813,0.817
4,Recursive 256,9905,0.737,0.870,0.914,0.970,0.737,0.795,0.806,0.813
5,Semantic,10594,0.734,0.849,0.899,0.953,0.734,0.787,0.799,0.806


In [14]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=display_df)

https://docs.google.com/spreadsheets/d/175S_8l-xukYnm-UNKI3MwVJjozXzN1SmcN6WsQthxKE/edit#gid=0


## 8. Save results

Both CSV (for spreadsheets / quick diffs) and JSON (for downstream scripts).

In [16]:
results_df.to_csv(EVAL_DIR / "retrieval_metrics.csv", index=False)

with open(EVAL_DIR / "retrieval_metrics.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved metrics to {EVAL_DIR / 'retrieval_metrics.csv'}")
print(f"Saved metrics to {EVAL_DIR / 'retrieval_metrics.json'}")

Saved metrics to eval/retrieval_metrics.csv
Saved metrics to eval/retrieval_metrics.json


## 9. Pick the winner

Sort by MRR@10 (a reasonable single-number summary of retrieval quality). The strategy at the top is the one to carry forward to whatever comes next — generation, reranking, or hybrid retrieval experiments.

In [15]:
ranked = display_df.sort_values("MRR@10", ascending=False).reset_index(drop=True)
print("Ranked by MRR@10:\n")
print(ranked[["Strategy", "Num chunks", "Hit@1", "Hit@5", "Hit@10", "MRR@10"]].to_string(index=False))

best = ranked.iloc[0]
print(f"\nBest strategy: {best['Strategy']}")
print(f"  Hit@10 = {best['Hit@10']:.3f}")
print(f"  MRR@10 = {best['MRR@10']:.3f}")

Ranked by MRR@10:

      Strategy  Num chunks  Hit@1  Hit@5  Hit@10  MRR@10
 Recursive 128       18638  0.749  0.923   0.950   0.817
Sentence-level       45952  0.737  0.929   0.967   0.813
 Recursive 256        9905  0.737  0.914   0.970   0.813
Document-level        5183  0.740  0.893   0.956   0.809
      Semantic       10594  0.734  0.899   0.953   0.806
     Fixed 128       17400  0.740  0.891   0.950   0.804

Best strategy: Recursive 128
  Hit@10 = 0.950
  MRR@10 = 0.817
